In [9]:
from pathlib import Path

import numpy as np
import optuna
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

from DL.Lab8.CustomSpeachDataset import CustomSpeachDataset
from DL.Lab8.SpeachClassifierModel import SpeachClassifierModel

In [10]:
device = device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [11]:
MODELS_DIR = Path("checkpoints")
MODELS_DIR.mkdir(exist_ok=True)

In [12]:
dataset = CustomSpeachDataset(Path("preprocessed_dataset_5"), preload=True)
dataset.to(device)

Preloading dataset from disk...


In [28]:
train_val_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y.cpu(),
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y.cpu()])

y_train = dataset.y.cpu()[train_val_indices]
train_val_dataset = torch.utils.data.Subset(dataset, train_val_indices)
train_val_sample_weights = all_sample_weights[train_val_indices]

test_dataset = torch.utils.data.Subset(dataset, test_indices)

len(train_val_dataset), len(y_train)

(8564, 8564)

In [32]:
dataset.y.unique()

tensor([0, 1, 2, 3, 4])

In [14]:
def train_new_model(writer: SummaryWriter, fold: int, params: dict, epochs: int, train_loader: DataLoader, val_loader: DataLoader, save_dir: Path|None, training_state: dict|None) -> SpeachClassifierModel:
    model = SpeachClassifierModel(params["dropout_rate"], output_classes=5)
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    best_val_loss = 99999999999.9

    loss_fn = nn.CrossEntropyLoss()

    start_epoch = 0

    if training_state:
        optimizer.load_state_dict(training_state["optimizer"])
        best_val_loss = training_state["best_val_loss"]
        start_epoch = training_state["epoch"]

    for epoch in range(start_epoch, epochs):
        print(f"Fold {fold} starting epoch {epoch}")

        model.train()
        train_loss = 0.0
        for data, target in train_loader:
            optimizer.zero_grad()

            y_pred = model(data)

            loss = loss_fn(y_pred, target)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                y_pred = model(data)
                loss = loss_fn(y_pred, target)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            if save_dir:
                save_path = save_dir / "best" / f"fold{fold}.pth"
                torch.save(model.state_dict(), save_path)

        writer.add_scalar(f"Loss/train/fold{fold}", train_loss, epoch)
        writer.add_scalar(f"Loss/val/fold{fold}", val_loss, epoch)

        if save_dir:
            torch.save(model.state_dict(), save_dir / "latest" / f"fold{fold}.pth")

            training_state = {
                "epoch": epoch,
                "optimizer": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "fold": fold,
            }
            torch.save(training_state, save_dir / "latest" / f"training_state.pth")

    model.best_val_loss = best_val_loss

    return model

In [15]:
def train_models_5cv(writer: SummaryWriter, params: dict, skf: StratifiedKFold, epochs: int, save_dir: Path|None, continue_training: bool = False) -> float:
    start_fold = 0

    if save_dir and continue_training:
        training_state = torch.load(save_dir / "latest" / f"training_state.pth")
        start_fold = training_state["fold"]
    else:
        training_state = None

    total_val_loss = 0.0

    for fold, (train_index, val_index) in enumerate(skf.split(y_train, y_train)):
        if fold < start_fold:
            print(f"fold {fold} skipped")
            continue

        print(f"starting training fold {fold}")

        train_sampler = WeightedRandomSampler(
            weights=train_val_sample_weights[train_index],
            num_samples=len(train_val_sample_weights[train_index]),
            replacement=True,
        )
        train_subset = torch.utils.data.Subset(train_val_dataset, train_index)
        train_loader = DataLoader(train_subset, batch_size=32, sampler=train_sampler)

        val_subset = torch.utils.data.Subset(train_val_dataset, val_index)
        val_loader = DataLoader(val_subset, batch_size=1024)

        model = train_new_model(writer, fold, params, epochs, train_loader, val_loader, save_dir, training_state)
        total_val_loss += model.best_val_loss

    avg_val_loss = total_val_loss / skf.n_splits
    return avg_val_loss


In [33]:
def objective_train(trial: optuna.trial.Trial) -> float:
    trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    trial.suggest_float("dropout_rate", 0.0, 0.5)
    trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    # save_dir = MODELS_DIR / "training"
    # save_dir.mkdir(parents=True, exist_ok=True)

    # (save_dir/"best").mkdir(parents=True, exist_ok=True)
    # (save_dir/"latest").mkdir(parents=True, exist_ok=True)
    # torch.save(params, save_dir / "best" / "model_params.pth")
    # torch.save(params, save_dir / "latest" / "model_params.pth")

    # Fewer epochs for hyperparameters optimization
    epochs = 1
    skf = StratifiedKFold(n_splits=5, shuffle=True)

    with SummaryWriter(f"runs/training/folds") as writer:
        avg_val_loss = train_models_5cv(writer, trial.params, skf, epochs, None, continue_training=False)

    metrics = {
        "hparams/avg_val_loss": avg_val_loss
    }

    with SummaryWriter(f"runs/optimization/weight_decay") as writer:
        writer.add_hparams(trial.params, metrics, run_name=f"trial_{trial.number}")
        writer.add_scalar("Loss/avg_val_loss", avg_val_loss, trial.number)

    return avg_val_loss

In [34]:
study = optuna.create_study(study_name="study_transfer_learning", direction="minimize")

[I 2026-05-21 17:36:04,751] A new study created in memory with name: study_transfer_learning


In [35]:
study.optimize(objective_train, n_trials=10)

starting training fold 0
Fold 0 starting epoch 0
starting training fold 1
Fold 1 starting epoch 0
starting training fold 2
Fold 2 starting epoch 0
starting training fold 3
Fold 3 starting epoch 0
starting training fold 4
Fold 4 starting epoch 0


[I 2026-05-21 17:37:40,763] Trial 0 finished with value: 1.3427050590515137 and parameters: {'lr': 0.0008732221651626526, 'dropout_rate': 0.42774094044405403, 'weight_decay': 0.00042005570907138187}. Best is trial 0 with value: 1.3427050590515137.


starting training fold 0
Fold 0 starting epoch 0
starting training fold 1
Fold 1 starting epoch 0
starting training fold 2
Fold 2 starting epoch 0
starting training fold 3
Fold 3 starting epoch 0


[W 2026-05-21 17:38:46,897] Trial 1 failed with parameters: {'lr': 7.75664928553498e-05, 'dropout_rate': 0.15277966254016567, 'weight_decay': 5.163868078413842e-05} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_6760/2562403739.py", line 19, in objective_train
    avg_val_loss = train_models_5cv(writer, trial.params, skf, epochs, None, continue_training=False)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6760/4165134449.py", line 30, in train_models_5cv
    model = train_new_model(writer, fold, params, epochs, train_loader, val_loader, save_dir, training_state)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [36]:
study.best_value, study.best_params

(1.3427050590515137,
 {'lr': 0.0008732221651626526,
  'dropout_rate': 0.42774094044405403,
  'weight_decay': 0.00042005570907138187})

In [ ]:
# Training with the best hparams
save_dir = MODELS_DIR / "training"
save_dir.mkdir(parents=True, exist_ok=True)

(save_dir/"best").mkdir(parents=True, exist_ok=True)
(save_dir/"latest").mkdir(parents=True, exist_ok=True)
torch.save(study.best_params, save_dir / "best" / "model_params.pth")
torch.save(study.best_params, save_dir / "latest" / "model_params.pth")

epochs = 100

train_sampler = WeightedRandomSampler(
    weights=train_val_sample_weights,
    num_samples=len(train_val_sample_weights),
    replacement=True,
)
train_loader = DataLoader(train_val_dataset, batch_size=32, sampler=train_sampler)
val_loader = DataLoader(test_dataset, batch_size=1024)

with SummaryWriter(f"runs/training/folds") as writer:
    avg_val_loss = train_new_model(writer, 9, study.best_params, epochs, train_loader, val_loader, save_dir, training_state=None)


Fold 9 starting epoch 0
Fold 9 starting epoch 1
Fold 9 starting epoch 2
Fold 9 starting epoch 3
